# Lesson 02 - Microsoft Agendi Raamistiku Uurimine

**Microsoft Agendi Raamistik (MAF)** on ühtne raamistik tehisintellekti agentide ehitamiseks. See pakub puhast, koostalituvat arhitektuuri nelja põhilise komponendiga:

- **Kliendi** – ühendub tehisintellekti mudeli otspunktiga ja haldab suhtlust
- **Agent** – ümbritseb klienti juhiste ja tööriistade definitsioonidega
- **Tööriistad** – laiendavad agendi võimekust kohandatud funktsioonidega, mida mudel saab kutsuda
- **Sessioon** – hoiab vestluse ajalugu mitme sammulisteks interaktsioonideks

Selles õppetükis ehitame **reisibroneerimise agendi**, mis kontrollib sihtkoha saadavust nende kontseptsioonide abil.


## Seadistus


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agendi raamistikuarhitektuuri mõistmine

Microsoft Agent Framework järgib kihilist arhitektuuri:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `AzureAIProjectAgentProvider` ühendub Azure OpenAI juurutusega. See haldab autentimist, päringu vormindamist ja vastuse parsimist.
2. **Agent** – Klientilt `provider.create_agent()` kaudu loodud agent kombineerib mudelile juurdepääsu juhistega (süsteemi prompt) ja tööriistadega.
3. **Tööriistad** – Python funktsioonid, mis on kaunistatud `@tool`, mida agent saab kutsuda toimingute tegemiseks või andmete pärimiseks.
4. **Seanss** – `AgentSession` objekt (loodud `agent.create_session()` kaudu), mis salvestab vestluse ajaloo, võimaldades mitme sammu dialoogi, kus agent mäletab eelnevat konteksti.

Ehitage iga kiht samm-sammult.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tööriistade lisamine koos @tool dekooraatoriga

Tööriistad võimaldavad agentidel tegutseda muul viisil kui lihtsalt teksti genereerimine. `@tool` dekooraator muudab tavalise Pythoni funktsiooni millekski, mida agent saab kutsuda.

Olulised punktid:
- Kasuta `Annotated[type, "kirjeldus"]`, et mudel mõistaks iga parameetrit.
- Docstring muutub tööriista kirjelduseks, mida mudel näeb.
- `approval_mode="never_require"` tähendab, et tööriist töötab automaatselt ilma kasutaja kinnitust küsimata.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agenti loomine tööriistadega

Nüüd ühendame kliendi, juhised ja tööriistad agendiks. `instructions` toimivad süsteemi juhisena — need määratlevad agendi isiksuse ja käitumise.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mitmevooruline vestlus koos sessioonidega

`AgentSession` (loodud `agent.create_session()` kaudu) hoiab kõiki sõnumeid vestluses. Samale sessioonile iga `agent.run()` kutsumise juures edasi andes on agendil ligipääs kogu vestluseajalool ja ta saab viidata varasematele sõnumitele.

Me anname `tools=[check_destination_availability]`, et agent saaks iga vooru ajal kutsuda meie saadavuse kontrollijat.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Kokkuvõte

Selles õppetükis uurisite Microsoft Agent Frameworki nelja alustala:

| Kontseptsioon | Mida õppisite |
|--------------|---------------|
| **Klient** | `AzureAIProjectAgentProvider` ühendub Azure OpenAI-ga volitustepõhise autentimisega |
| **Agent** | `provider.create_agent()` seob mudeliühenduse juhiste ja nimega |
| **Tööriistad** | `@tool` dekoratiiv võimaldab agendil kutsuda Pythoni funktsioone |
| **Seanss** | `agent.create_session()` haldab vestluse ajalugu mitme pöörde jooksul |

Need ehituskivid moodustavad koos agente, kes suudavad pidada loomulikke vestlusi, kutsuda väliseid funktsioone ja hoida konteksti — see on aluseks keerukamate agentuursete mustrite jaoks tulevastes õppetundides.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud tehisintellekti tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüame täpsust tagada, palun arvestage, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Algkeeles dokumenti tuleks pidada autoriteetseks allikaks. Tähtsa teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta käesoleva tõlke kasutamisest tulenevate arusaamatuste või valesti tõlgendamise eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
